# Fase 2: Indicadores Técnicos Adaptativos e Microestrutura (Dollar Bars)

Este notebook demonstra o fluxo de análise técnica quantitativa e amostragem de microestrutura de mercado:
1. **McGinley Dynamic Indicator**: Filtro adaptativo de preço que ajusta automaticamente a sua velocidade de rastreamento com base na volatilidade do ativo.
2. **Cross-Sectional Volatility-Adjusted Momentum**: Ranking percentil $(0.0 \text{ a } 1.0)$ do momentum ajustado pela volatilidade em janelas de 1M (21d), 3M (63d) e 12M (252d).
3. **Amostragem por Dollar Bars**: Agrupamento de ticks de alta frequência baseado no volume financeiro transacionado (preço $\times$ volume), reduzindo a heterocedasticidade estatística das séries temporais.

### Formulações Matemáticas
- **McGinley Dynamic**:
  $$M_t = M_{t-1} + \frac{P_t - M_{t-1}}{k \cdot N \cdot \left(\frac{P_t}{M_{t-1}}\right)^4}$$
- **Momentum Ajustado pela Volatilidade**:
  $$\text{Vol-Adj Mom}_{\tau} = \frac{R_{\tau}}{\sigma_{\tau} \sqrt{252}}$$
- **Dollar Bars Condition**:
  $$\sum_{i=1}^{k} P_i \cdot V_i \ge \text{Threshold}$$

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Adicionar diretório raiz ao path
sys.path.insert(0, os.path.abspath('..'))

from src.features.technical import (
    mcginley_dynamic,
    compute_volatility_adjusted_momentum,
    compute_cross_sectional_ranks,
    build_dollar_bars,
)

## 1. Demonstração do McGinley Dynamic vs Médias Móveis Tradicionais

In [2]:
np.random.seed(42)
n_days = 250
dates = pd.date_range("2025-01-01", periods=n_days, freq="B")

# Simulação de preços com saltos e tendência
returns = np.random.normal(0.0008, 0.015, size=n_days)
returns[80:90] -= 0.03  # Queda brusca
returns[150:165] += 0.025 # Rally repentino
prices = pd.Series(100.0 * np.exp(np.cumsum(returns)), index=dates, name="Close")

# Cálculo do McGinley Dynamic e SMA(14) para comparação
mgd_14 = mcginley_dynamic(prices, n=14, k=0.6)
sma_14 = prices.rolling(14).mean()

df_plot = pd.DataFrame({
    "Preço de Fecho": prices,
    "McGinley Dynamic (N=14)": mgd_14,
    "SMA (14)": sma_14
})

print("Últimos 5 valores:")
print(df_plot.tail())

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(df_plot.index, df_plot["Preço de Fecho"], label="Preço de Fecho", color="black", alpha=0.6, lw=1.5)
ax.plot(df_plot.index, df_plot["McGinley Dynamic (N=14)"], label="McGinley Dynamic (Adaptativo)", color="blue", lw=2)
ax.plot(df_plot.index, df_plot["SMA (14)"], label="Média Móvel Simples SMA(14)", color="orange", linestyle="--", lw=1.5)
ax.set_title("McGinley Dynamic vs SMA(14) - Rastreamento Adaptativo de Tendência", fontsize=13)
ax.set_ylabel("Preço ($)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 2. Momentum Cross-Sectional Ajustado pela Volatilidade (1M, 3M, 12M)

In [3]:
# Universo de 5 ativos com regimes distintos
np.random.seed(101)
univ_returns = {
    "AAPL": np.random.normal(0.0012, 0.012, size=300),
    "MSFT": np.random.normal(0.0010, 0.011, size=300),
    "GOOGL": np.random.normal(0.0006, 0.014, size=300),
    "AMZN": np.random.normal(0.0002, 0.018, size=300),
    "TSLA": np.random.normal(-0.0005, 0.025, size=300),
}
dates_panel = pd.date_range("2024-01-01", periods=300, freq="B")
univ_prices = pd.DataFrame({
    ticker: 100.0 * np.exp(np.cumsum(ret)) for ticker, ret in univ_returns.items()
}, index=dates_panel)

# Ranking Cross-Sectional nos horizontes de 21d, 63d, 252d
ranks = compute_cross_sectional_ranks(univ_prices, windows=(21, 63, 252), weights=(0.2, 0.3, 0.5))

print("=== RANKINGS COMPÓSITOS CROSS-SECTIONAL RECENTES (0.0 a 1.0) ===")
print(ranks["composite_rank"].tail())

## 3. Amostragem de Microestrutura: Geração de Dollar Bars

In [4]:
# Simulação de 1.000 transações de alta frequência (Ticks)
np.random.seed(7)
n_ticks = 1000
base_time = pd.Timestamp("2026-03-01 09:30:00")
tick_times = [base_time + pd.Timedelta(seconds=int(i * 3 + np.random.uniform(0, 2))) for i in range(n_ticks)]
tick_prices = 150.0 + np.cumsum(np.random.normal(0, 0.1, n_ticks))
tick_volumes = np.random.exponential(scale=200, size=n_ticks) + 10

df_ticks = pd.DataFrame({
    "timestamp": tick_times,
    "price": tick_prices,
    "volume": tick_volumes
})

# Limite financeiro por barra: $100.000 de volume financeiro acumulado
dollar_threshold = 100000.0
dollar_bars = build_dollar_bars(df_ticks, threshold=dollar_threshold)

print(f"Total de Ticks processados: {len(df_ticks)}")
print(f"Total de Dollar Bars geradas (Threshold = ${dollar_threshold:,.0f}): {len(dollar_bars)}")
print("\nPrimeiras 5 Dollar Bars consolidadas (OHLCV + VWAP):")
print(dollar_bars[['timestamp_start', 'open', 'high', 'low', 'close', 'volume', 'dollar_volume', 'vwap', 'tick_count']].head())